In [1]:
%pip install metapub cssselect requests


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import time
import requests
from metapub import PubMedFetcher, FindIt

/Users/luran/Desktop/T3/TER/.venv/lib/python3.11/site-packages/eutils/__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-02-09 23:32:46 lurandendiannao metapub.config[46593] WARNING NCBI_API_KEY was not set.


In [5]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Connection": "keep-alive"
}

In [8]:
TOPICS = {
    "Diagnosis": "Sepsis[Title] AND (Criteria[Title] OR Definition[Title] OR Diagnosis[Title])",
    "Treatment": "Sepsis[Title] AND (Treatment[Title] OR Therapy[Title] OR Management[Title])"
}

MAX_PAPERS = 20

In [9]:
def download_papers(topic_name, query):
    # 1. 创建保存目录
    save_dir = f"Sepsis_Papers/{topic_name}"
    os.makedirs(save_dir, exist_ok=True)
    
    fetch = PubMedFetcher()
    print(f"\n🔍 [Topic: {topic_name}] Searching: {query}")
    
    try:
        # 获取 PMIDs
        pmids = fetch.pmids_for_query(query, retmax=MAX_PAPERS)
        print(f"✅ Found {len(pmids)} papers.")
    except Exception as e:
        print(f"❌ Search failed: {e}")
        return

    # 2. 遍历下载
    for i, pmid in enumerate(pmids):
        print(f"\n[{i+1}/{len(pmids)}] PMID: {pmid}")
        file_path = os.path.join(save_dir, f"{pmid}.pdf")
        
        # 跳过已存在的
        if os.path.exists(file_path):
            # 简单检查一下文件大小，防止之前下载了损坏的空文件
            if os.path.getsize(file_path) > 1024: 
                print("    ⏭️  已存在 (跳过)")
                continue
            else:
                print("    ⚠️  文件存在但过小，重新下载...")

        try:
            # --- 核心修复 1: 更加鲁棒的链接获取 ---
            source = FindIt(pmid)
            url = source.url

            # 如果主 url 为空，尝试找 PMC 的链接 (通常更稳)
            if not url:
                if hasattr(source, 'pmc_url') and source.pmc_url:
                    print("    🔄 切换至 PMC 镜像源...")
                    url = source.pmc_url
            
            if not url:
                print("    ❌ 无可用下载链接 (可能是付费墙)")
                continue
            
            print(f"    🔗 链接: {url}")

            # --- 核心修复 2: 下载流与类型检查 ---
            r = requests.get(url, headers=HEADERS, stream=True, allow_redirects=True, timeout=30)
            
            if r.status_code == 200:
                # 检查 Content-Type，防止把 HTML 网页存成 PDF
                content_type = r.headers.get('Content-Type', '').lower()
                if 'pdf' not in content_type and 'octet-stream' not in content_type:
                    print(f"    ⚠️  下载内容的不是PDF (是 {content_type})，跳过")
                    continue

                with open(file_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=1024*1024): 
                        if chunk:
                            f.write(chunk)
                
                print("    🎉 下载成功")
            else:
                print(f"    ⚠️  失败 (状态码 {r.status_code})")

        except Exception as e:
            print(f"    ❌ 错误: {e}")
        
        # 礼貌性延时，防封 IP
        time.sleep(2)

def main():
    print("🚀 开始双路检索任务...")
    for topic, query in TOPICS.items():
        download_papers(topic, query)
    print("\n🏁 所有任务结束！")

if __name__ == "__main__":
    main()

🚀 开始双路检索任务...

🔍 [Topic: Diagnosis] Searching: Sepsis[Title] AND (Criteria[Title] OR Definition[Title] OR Diagnosis[Title])
✅ Found 20 papers.

[1/20] PMID: 38245897
    🔗 链接: https://jamanetwork.com/journals/jama/articlepdf/2814296/jama_sanchezpinto_2024_oi_240003_1709591810.56162.pdf
    ⚠️  失败 (状态码 403)

[2/20] PMID: 32227831
    ❌ 无可用下载链接 (可能是付费墙)

[3/20] PMID: 36304809
    ⏭️  已存在 (跳过)

[4/20] PMID: 38245889
    🔗 链接: https://jamanetwork.com/journals/jama/articlepdf/2814297/jama_schlapbach_2024_oi_240002_1711118634.86488.pdf
    ⚠️  失败 (状态码 403)

[5/20] PMID: 38084084
    ⏭️  已存在 (跳过)

[6/20] PMID: 34612847
    ⏭️  已存在 (跳过)

[7/20] PMID: 38968960
    ❌ 无可用下载链接 (可能是付费墙)

[8/20] PMID: 37904710
    ⏭️  已存在 (跳过)

[9/20] PMID: 36878489
    ❌ 无可用下载链接 (可能是付费墙)

[10/20] PMID: 29626019
    ⏭️  已存在 (跳过)

[11/20] PMID: 26903335


2026-02-10 00:06:59 lurandendiannao metapub.DxDOI[46593] INFO URL returned status code 403: https://jamanetwork.com/article.aspx?doi=10.1001/jama.2016.0288
2026-02-10 00:06:59 lurandendiannao metapub.DxDOI[46593] INFO cached results for key 10.1001/jama.2016.0288 (10.1001/jama.2016.0288) 


    ❌ 无可用下载链接 (可能是付费墙)

[12/20] PMID: 26903336
    🔗 链接: http://europepmc.org/backend/ptpmcrender.fcgi?accid=PMC4910392&blobtype=pdf
    🎉 下载成功

[13/20] PMID: 39676169
    🔗 链接: http://europepmc.org/backend/ptpmcrender.fcgi?accid=PMC11648306&blobtype=pdf
    🎉 下载成功

[14/20] PMID: 40384873
    🔗 链接: http://europepmc.org/backend/ptpmcrender.fcgi?accid=PMC12080397&blobtype=pdf
    🎉 下载成功

[15/20] PMID: 32699026
    🔗 链接: http://europepmc.org/backend/ptpmcrender.fcgi?accid=PMC9489194&blobtype=pdf
    🎉 下载成功

[16/20] PMID: 33186343
    ❌ 无可用下载链接 (可能是付费墙)

[17/20] PMID: 30915218
    🔗 链接: http://europepmc.org/backend/ptpmcrender.fcgi?accid=PMC6429642&blobtype=pdf
    🎉 下载成功

[18/20] PMID: 28963294
    🔗 链接: http://europepmc.org/backend/ptpmcrender.fcgi?accid=PMC5623518&blobtype=pdf
    🎉 下载成功

[19/20] PMID: 34030810
    ❌ 无可用下载链接 (可能是付费墙)

[20/20] PMID: 39256242
    🔗 链接: http://europepmc.org/backend/ptpmcrender.fcgi?accid=PMC11473534&blobtype=pdf
    🎉 下载成功

🔍 [Topic: Treatment] Searching: 

2026-02-10 00:07:38 lurandendiannao metapub.DxDOI[46593] INFO URL returned status code 403: https://www.acpjournals.org/doi/10.7326/ANNALS-25-02685
2026-02-10 00:07:38 lurandendiannao metapub.DxDOI[46593] INFO cached results for key 10.7326/ANNALS-25-02685 (10.7326/ANNALS-25-02685) 
2026-02-10 00:07:38 lurandendiannao metapub.findit.handlers[46593] ERROR Dance function the_sciencedirect_disco failed for PMID 41213152, journal 'Ann Intern Med', publisher 'sciencedirect': MISSING: ScienceDirect article could not be accessed via DOI fallback - Journal: Ann Intern Med


    ❌ 无可用下载链接 (可能是付费墙)

[16/20] PMID: 40685448
    🔗 链接: http://europepmc.org/backend/ptpmcrender.fcgi?accid=PMC12277237&blobtype=pdf
    🎉 下载成功

[17/20] PMID: 34340174
    🔗 链接: http://europepmc.org/backend/ptpmcrender.fcgi?accid=PMC8541922&blobtype=pdf
    🎉 下载成功

[18/20] PMID: 30824138
    ❌ 无可用下载链接 (可能是付费墙)

[19/20] PMID: 32004193


2026-02-10 00:07:53 lurandendiannao httpx[46593] INFO HTTP Request: GET https://api.crossref.org/works/10.1097/MCC.0000000000000696 "HTTP/1.1 200 OK"
2026-02-10 00:07:56 lurandendiannao metapub.findit.handlers[46593] ERROR Dance function the_wolterskluwer_volta failed for PMID 32004193, journal 'Curr Opin Crit Care', publisher 'Wolterskluwer': BLOCKED: All Wolters Kluwer URL patterns failed for DOI 10.1097/MCC.0000000000000696 (Last error: DENIED: Wolters Kluwer url (https://doi.org/10.1097/MCC.0000000000000696) returned non-PDF content)


    ❌ 无可用下载链接 (可能是付费墙)

[20/20] PMID: 33236524
    🔗 链接: http://europepmc.org/backend/ptpmcrender.fcgi?accid=PMC11469323&blobtype=pdf
    🎉 下载成功

🏁 所有任务结束！


# 2.每篇论文的逻辑链条

In [16]:
%pip install torch transformers accelerate pdfplumber


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [27]:
import os
import json
import torch
import pdfplumber
from transformers import AutoTokenizer, AutoModelForCausalLM

In [34]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. 检测设备
# 优先使用 MPS (Mac GPU)，其次 CUDA (Nvidia)，最后 CPU
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"🚀 准备运行在: {device}")

# 2. 设置模型路径 (如果你还没下载，保持 ID 不变)
MODEL_ID = "google/gemma-2b-it"

print("⏳ 正在加载模型...")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    
    # --- 核心修复 ---
    # 如果是 Mac (mps)，通常不支持 device_map="auto" 这种自动切分
    # 我们直接把整个模型加载到 mps 上
    if device == "mps":
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, 
            torch_dtype=torch.float16
        ).to("mps") # 显式移动到 MPS
    else:
        # 对于 CUDA 或 CPU，保持原有逻辑
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, 
            torch_dtype=torch.float16, 
            device_map="auto"
        )
        
    print(f"✅ 模型成功加载到了 {model.device}！")

except Exception as e:
    print(f"❌ 加载失败: {e}")

🚀 准备运行在: mps
⏳ 正在加载模型...


Loading checkpoint shards: 100%|██████████| 2/2 [00:20<00:00, 10.32s/it]


✅ 模型成功加载到了 mps:0！


In [35]:
# 测试一下
import json

# 1. 定义一个简单的 Prompt 模板
# 我们先用一段短文本测试，不涉及 PDF 读取
test_text = """
Introduction: Sepsis is a life-threatening organ dysfunction.
Methods: We observed that patients with Lactate > 4mmol/L had higher mortality.
Results: Administration of Hydrocortisone reduced mortality in septic shock patients.
"""

prompt = f"""
You are a medical logic extractor. 
Extract logical chains from the text below as a strictly valid JSON list.
Format: [Source] -> [Relation] -> [Target].
Relations allowed: INDICATES, CAUSES, TREATED_WITH, REDUCES.

Text: {test_text}

JSON Output:
"""

# 2. 编码 (Input)
# 注意：不需要再加 .to(device)，因为 model 已经在 mps 上了，
# 但 input_ids 必须手动挪到 mps 上
inputs = tokenizer(prompt, return_tensors="pt").to("mps")

print("🧠 Gemma 正在思考 (第一次运行可能稍慢)...")

# 3. 生成 (Inference)
outputs = model.generate(
    **inputs, 
    max_new_tokens=256,   # 测试用，不用太长
    temperature=0.1,      # 低温，保证稳定
    do_sample=True
)

# 4. 解码 (Decode)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# 5. 提取 JSON
try:
    # 截取 "JSON Output:" 之后的内容
    response_part = generated_text.split("JSON Output:")[-1]
    
    # 寻找 JSON 数组的方括号 []
    start_idx = response_part.find("[")
    end_idx = response_part.rfind("]") + 1
    
    if start_idx != -1 and end_idx != -1:
        json_str = response_part[start_idx:end_idx]
        result = json.loads(json_str)
        print("\n✅ 测试通过！Gemma 提取到了以下逻辑：")
        print(json.dumps(result, indent=4, ensure_ascii=False))
    else:
        print("⚠️ 未找到 JSON 格式，原始输出如下：")
        print(response_part)

except Exception as e:
    print(f"❌ JSON 解析错误: {e}")
    print("完整输出:", generated_text)

🧠 Gemma 正在思考 (第一次运行可能稍慢)...

✅ 测试通过！Gemma 提取到了以下逻辑：
[
    {
        "Source": "Introduction",
        "Relation": "causes",
        "Target": "Sepsis"
    },
    {
        "Source": "Methods",
        "Relation": "indicates",
        "Target": "Lactate > 4mmol/L"
    },
    {
        "Source": "Methods",
        "Relation": "causes",
        "Target": "Sepsis"
    },
    {
        "Source": "Results",
        "Relation": "reduces",
        "Target": "Mortality"
    }
]


In [37]:
%pip install pymupdf pdfplumber pypdf

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.5/22.5 MB 3.2 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pymupdf]m1/2 [pymupdf]

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [39]:
import os

# 修改这里：换成你的其中一个报错 PDF 的路径
file_path = "Sepsis_Papers/Diagnosis/39676169.pdf" 

print(f"🔍 正在深度诊断文件: {file_path}")
print("-" * 50)

# --- 测试 1: 使用 PyMuPDF (fitz) ---
# 这是目前最强、最快的库，如果它读不出来，大概率是编码问题
try:
    import fitz
    print("\n[测试 1] PyMuPDF (fitz) 引擎:")
    doc = fitz.open(file_path)
    text = ""
    for page in doc:
        text += page.get_text()
    
    clean_text = text.strip()
    print(f"   - 提取字符数: {len(clean_text)}")
    print(f"   - 前 100 个字符预览: {repr(clean_text[:100])}")
    
    if len(clean_text) == 0:
        print("   ❌ 结果为空字符串")
    elif "(cid:" in clean_text:
        print("   ⚠️ 警告: 发现 CID 字体乱码 (如 (cid:123))，这是编码缺失问题")
    else:
        print("   ✅ 提取看似正常")
except Exception as e:
    print(f"   ❌ 报错: {e}")


# --- 测试 2: 使用 pdfplumber ---
# 这个库擅长处理复杂的布局，但比较慢
try:
    import pdfplumber
    print("\n[测试 2] pdfplumber 引擎:")
    with pdfplumber.open(file_path) as pdf:
        text = ""
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text
                
    clean_text = text.strip()
    print(f"   - 提取字符数: {len(clean_text)}")
    print(f"   - 前 100 个字符预览: {repr(clean_text[:100])}")
    
    if len(clean_text) == 0:
        print("   ❌ 结果为空字符串")
    else:
        print("   ✅ 提取看似正常")
except Exception as e:
    print(f"   ❌ 报错: {e}")


# --- 测试 3: 使用 pypdf (原 PyPDF2) ---
# 这是最老牌的库，经常因为版本或标准问题读不出东西
try:
    from pypdf import PdfReader
    print("\n[测试 3] pypdf 引擎:")
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted

    clean_text = text.strip()
    print(f"   - 提取字符数: {len(clean_text)}")
    print(f"   - 前 100 个字符预览: {repr(clean_text[:100])}")
    
    if len(clean_text) == 0:
        print("   ❌ 结果为空字符串")
    else:
        print("   ✅ 提取看似正常")
except Exception as e:
    print(f"   ❌ 报错: {e}")

print("-" * 50)

🔍 正在深度诊断文件: Sepsis_Papers/Diagnosis/39676169.pdf
--------------------------------------------------

[测试 1] PyMuPDF (fitz) 引擎:
   - 提取字符数: 156907
   - 前 100 个字符预览: 'Xu\xa0et\xa0al. Military Medical Research           (2024) 11:78  \nhttps://doi.org/10.1186/s40779-024-0058'
   ✅ 提取看似正常

[测试 2] pdfplumber 引擎:
   - 提取字符数: 153908
   - 前 100 个字符预览: 'Xu et al. Military Medical Research (2024) 11:78\nhttps://doi.org/10.1186/s40779-024-00581-0\nREVIEW O'
   ✅ 提取看似正常

[测试 3] pypdf 引擎:
   - 提取字符数: 156616
   - 前 100 个字符预览: 'Xu\xa0et\xa0al. Military Medical Research           (2024) 11:78  \nhttps://doi.org/10.1186/s40779-024-0058'
   ✅ 提取看似正常
--------------------------------------------------


In [45]:
%pip install openpyxl

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [46]:
import os
import fitz  # PyMuPDF
import re
import json
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ================= Configuration =================
MODEL_ID = "google/gemma-2b-it"
ROOT_DIR = "Sepsis_Papers"

# ================= 1. Load Model (Offline) =================
print(f"⏳ Loading model: {MODEL_ID} ...")
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"🖥️  Compute Device: {device.upper()}")

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, 
        torch_dtype=torch.float16,
        device_map=device,
        local_files_only=True 
    )
    print("✅ Model loaded successfully!")
except Exception as e:
    print(f"❌ Model loading failed: {e}")
    raise e 

# ================= 2. Helper Functions (含 read_pdf) =================

def clean_text(text):
    if not text: return ""
    text = text.replace(u'\xa0', ' ').replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def read_pdf(file_path):
    """
    补上的读取函数：读取 PDF 所有页面并清洗
    """
    try:
        doc = fitz.open(file_path)
        full_text = " ".join([page.get_text() for page in doc])
        doc.close()
        return clean_text(full_text)
    except Exception as e:
        print(f"   ❌ Read Error: {os.path.basename(file_path)} -> {e}")
        return None

def extract_content_from_abstract(text, max_length=2000):
    """Start from 'Abstract' to capture the core logic."""
    match = re.search(r'(abstract|introduction)', text, re.IGNORECASE)
    if match:
        start_index = match.start()
        return text[start_index : start_index + max_length]
    else:
        return text[:max_length]

def extract_json_content(text):
    try:
        return json.loads(text)
    except:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            try: return json.loads(match.group())
            except: pass
    return None

def analyze_logic_chain(filename, text_snippet):
    """
    Prompt: Focus on Diagnostic Logic Chain.
    """
    chat = [
        { "role": "user", "content": f"""
You are a medical AI researcher. Analyze the following paper snippet and extract the **Diagnostic Logic Chain**.

Paper Snippet:
{text_snippet}

Task: Extract the following fields into a valid JSON object:
1. "inputs": What clinical features are used? (e.g., Vital signs, HR, WBC, Lactate).
2. "process": How does the method process these inputs? (e.g., "Extracts temporal patterns using LSTM", "Applies thresholds to SOFA score").
3. "output": What is the specific prediction target? (e.g., "Sepsis onset in 6 hours", "Mortality risk").
4. "logic_summary": A one-sentence summary of the logic: "Input A -> Algorithm B -> Prediction C".

Constraint: Output ONLY the JSON string. No Markdown.
""" }
    ]
    
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=500, do_sample=False)
        
    output_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    return extract_json_content(output_text), output_text

# ================= 3. Execution =================

all_results = []
print("-" * 60)
print("🚀 Starting Logic Chain Extraction...")

for root, dirs, files in os.walk(ROOT_DIR):
    category = os.path.basename(root)
    pdf_files = [f for f in files if f.lower().endswith('.pdf')]
    
    if not pdf_files: continue
    
    print(f"\n📂 Category: {category}")
    
    for f in pdf_files:
        path = os.path.join(root, f)
        
        # Read
        raw_text = read_pdf(path)
        if not raw_text or len(raw_text) < 500: continue
            
        print(f"   Analyzing Logic: {f} ... ", end="")
        
        # Cut (From Abstract)
        processed_content = extract_content_from_abstract(raw_text, max_length=2000)
        
        # Extract Logic
        json_data, raw_output = analyze_logic_chain(f, processed_content)
        
        if json_data:
            json_data['filename'] = f
            json_data['category'] = category
            all_results.append(json_data)
            print(" ✅")
        else:
            print(" ⚠️ JSON Failed")
            # Save raw text to debug
            all_results.append({
                'filename': f,
                'inputs': 'Parse Error',
                'logic_summary': raw_output 
            })

# ================= 4. Save =================

if all_results:
    df = pd.DataFrame(all_results)
    # Reorder
    cols = ['filename', 'inputs', 'process', 'output', 'logic_summary']
    final_cols = [c for c in cols if c in df.columns]
    df = df[final_cols]
    
    output_file = "Sepsis_Logic_Chain.xlsx"
    df.to_excel(output_file, index=False)
    print(f"\n🎉 Done! Logic chains saved to {output_file}")
    display(df.head())

⏳ Loading model: google/gemma-2b-it ...
🖥️  Compute Device: MPS


Loading checkpoint shards: 100%|██████████| 2/2 [00:49<00:00, 24.70s/it]


✅ Model loaded successfully!
------------------------------------------------------------
🚀 Starting Logic Chain Extraction...

📂 Category: Treatment
   Analyzing Logic: 39302568.pdf ...  ✅
   Analyzing Logic: 31747556.pdf ...  ✅
   Analyzing Logic: 34340174.pdf ...  ✅
   Analyzing Logic: 33236524.pdf ...  ✅
   Analyzing Logic: 38577320.pdf ...  ✅
   Analyzing Logic: 30915218.pdf ...  ✅
   Analyzing Logic: 40685448.pdf ...  ✅
   Analyzing Logic: 29626019.pdf ...  ✅
   Analyzing Logic: 34446092.pdf ...  ⚠️ JSON Failed
   Analyzing Logic: 29022217.pdf ...  ⚠️ JSON Failed
   Analyzing Logic: 38084084.pdf ...  ✅
   Analyzing Logic: 25834828.pdf ...  ✅
   Analyzing Logic: 32978579.pdf ...  ✅

📂 Category: Diagnosis
   Analyzing Logic: 39676169.pdf ...  ✅
   Analyzing Logic: 37904710.pdf ...  ✅
   Analyzing Logic: 32699026.pdf ...  ⚠️ JSON Failed
   Analyzing Logic: 36304809.pdf ...  ✅
   Analyzing Logic: 30915218.pdf ...  ✅
   Analyzing Logic: 29626019.pdf ...  ✅
   Analyzing Logic: 34612847

,filename,inputs,process,output,logic_summary
0,39302568.pdf,"[Vital signs, HR, WBC, Lactate]",Extracts temporal patterns using LSTM,Sepsis onset in 6 hours,Input vital signs and lab values over time to ...
1,31747556.pdf,None,None,Sepsis onset in 6 hours,"Input vital signs, HR, WBC, and lactate, and a..."
2,34340174.pdf,None,None,Sepsis onset in 6 hours,Input A -> Algorithm B -> Prediction C
3,33236524.pdf,None,None,Sepsis onset in 6 hours,Input A -> Algorithm B -> Prediction C
4,38577320.pdf,"[Vital signs, HR, WBC, Lactate]",Extracts temporal patterns using LSTM,Sepsis onset in 6 hours,Input vital signs and lab values over time to ...


# ---------------------------------------

# 把收集到的论文按照section处理

In [26]:
%pip install pdfplumber pandas

  Using cached pandas-3.0.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached pillow-12.1.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached cffi-2.0.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 5.2 MB/s  0:00:01 eta 0:00:01
Using cached pandas-3.0.0-cp311-cp311-macosx_11_0_arm64.whl (9.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 7.9 MB/s  0:00:00 eta 0:00:01
Using cached cffi-2.0.0-cp311-cp311-macosx_11_0_arm64.whl (180 kB)
Using cached pillow-12.1.0-cp311-cp311-macosx_11_0_arm64.whl (4.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 10.4 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [pdfplumber]8 [pdfminer.six]
Note: you may need to restart the kernel to use updated packages.


In [27]:
import pdfplumber
import pandas as pd

In [28]:
SOURCE_DIR = "Annane_Papers_Collection"
OUTPUT_DIR = "Sepsis_Datasets"

In [29]:
# 针对 Introduction / Background
PAT_INTRO = r"(?i)\n\s*(introduction|background|objectives?)\s*\n"
# 针对 Methods / Materials
PAT_METHOD = r"(?i)\n\s*(methods?|materials?|methodology|patients? and methods?|study design)\s*\n"
# 针对 Results (用来截断 Method)
PAT_RESULT = r"(?i)\n\s*(results?|findings?)\s*\n"
# 针对 Discussion / Conclusion
PAT_CONCLU = r"(?i)\n\s*(discussion|conclusions?|summary)\s*\n"
# 针对 References (用来截断 Conclusion)
PAT_REF = r"(?i)\n\s*(references?|bibliography|acknowledgements?)\s*\n"

In [30]:
def clean_text(text):
    """简单的文本清洗：去除多余换行，保留段落感"""
    if not text: return ""
    # 把连续的换行换成空格，但保留段落
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [31]:
def split_paper_sections(text):
    """
    核心逻辑：寻找关键词在文本中的位置(index)，然后进行切片(Slicing)
    """
    sections = {
        "Abstract_Set": "",
        "Method_Set": "",
        "Conclusion_Set": ""
    }
    
    # 1. 寻找各个锚点的位置
    # search() 只找第一个匹配项
    m_intro = re.search(PAT_INTRO, text)
    m_method = re.search(PAT_METHOD, text)
    m_result = re.search(PAT_RESULT, text)
    m_conclu = re.search(PAT_CONCLU, text)
    m_ref = re.search(PAT_REF, text)

    # 获取索引位置，如果没有找到，设为 -1
    idx_intro = m_intro.start() if m_intro else -1
    idx_method = m_method.start() if m_method else -1
    idx_result = m_result.start() if m_result else -1
    idx_conclu = m_conclu.start() if m_conclu else -1
    idx_ref = m_ref.start() if m_ref else len(text)

    # 2. 切分【Abstract/Introduction 数据集】
    # 逻辑：从头开始，直到 Method 出现之前。如果没有 Method，就到 Result 之前。
    end_of_intro = idx_method if idx_method != -1 else idx_result
    if end_of_intro != -1:
        sections["Abstract_Set"] = text[:end_of_intro]
    else:
        # 如果啥标题都没找到，取前 20%
        sections["Abstract_Set"] = text[:int(len(text)*0.2)]

    # 3. 切分【Method 数据集】
    # 逻辑：从 Method 开始，到 Result 之前
    if idx_method != -1:
        end_of_method = idx_result if idx_result != -1 else idx_conclu
        if end_of_method != -1 and end_of_method > idx_method:
            sections["Method_Set"] = text[idx_method:end_of_method]
        else:
            # 只有开始没有结束，取后 2000 字符
            sections["Method_Set"] = text[idx_method:idx_method+2000]
    
    # 4. 切分【Conclusion 数据集】
    # 逻辑：从 Discussion/Conclusion 开始，直到 References
    if idx_conclu != -1:
        sections["Conclusion_Set"] = text[idx_conclu:idx_ref]
    elif idx_result != -1:
        # 如果没找到 Conclusion 标题，可能包含在 Result 里，取 Result 后半部分
        sections["Conclusion_Set"] = text[idx_result:idx_ref]

    # 清洗一下数据
    for k, v in sections.items():
        sections[k] = clean_text(v)
        
    return sections

In [32]:
def main():
    if not os.path.exists(OUTPUT_DIR):
        os.mkdir(OUTPUT_DIR)
        
    data_list = []
    
    files = [f for f in os.listdir(SOURCE_DIR) if f.endswith(".pdf")]
    print(f"🚀 开始处理 {len(files)} 篇 PDF...")

    for i, file in enumerate(files):
        path = os.path.join(SOURCE_DIR, file)
        print(f"[{i+1}/{len(files)}] 解析: {file}")
        
        full_text = ""
        try:
            # 读取 PDF 全部文本
            with pdfplumber.open(path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        full_text += "\n" + page_text
            
            # 核心切分
            parts = split_paper_sections(full_text)
            parts["filename"] = file
            data_list.append(parts)
            
        except Exception as e:
            print(f"    ❌ 读取失败: {e}")

    # 转成 DataFrame
    df = pd.DataFrame(data_list)

    # 分别保存三个数据集
    print("\n💾 正在保存三个数据集...")
    
    # Dataset 1: Abstract/Intro
    df_abs = df[["filename", "Abstract_Set"]]
    df_abs.to_csv(os.path.join(OUTPUT_DIR, "Dataset_Abstract.csv"), index=False)
    print(f"   -> Created: Dataset_Abstract.csv ({len(df_abs)} entries)")

    # Dataset 2: Method
    df_method = df[["filename", "Method_Set"]]
    df_method.to_csv(os.path.join(OUTPUT_DIR, "Dataset_Method.csv"), index=False)
    print(f"   -> Created: Dataset_Method.csv ({len(df_method)} entries)")

    # Dataset 3: Conclusion
    df_conc = df[["filename", "Conclusion_Set"]]
    df_conc.to_csv(os.path.join(OUTPUT_DIR, "Dataset_Conclusion.csv"), index=False)
    print(f"   -> Created: Dataset_Conclusion.csv ({len(df_conc)} entries)")

if __name__ == "__main__":
    main()

🚀 开始处理 19 篇 PDF...
[1/19] 解析: 34351722.pdf
[2/19] 解析: 35732834.pdf
[3/19] 解析: 26903338.pdf
[4/19] 解析: 41180093.pdf
[5/19] 解析: 26633262.pdf
[6/19] 解析: 29025194.pdf
[7/19] 解析: 33159530.pdf
[8/19] 解析: 37176780.pdf
[9/19] 解析: 33150472.pdf
[10/19] 解析: 34789298.pdf
[11/19] 解析: 30111341.pdf
[12/19] 解析: 10798738.pdf
[13/19] 解析: 38250247.pdf
[14/19] 解析: 33876268.pdf
[15/19] 解析: 30105022.pdf
[16/19] 解析: 39609770.pdf
[17/19] 解析: 37888913.pdf
[18/19] 解析: 23361625.pdf
[19/19] 解析: 39787606.pdf

💾 正在保存三个数据集...
   -> Created: Dataset_Abstract.csv (19 entries)
   -> Created: Dataset_Method.csv (19 entries)
   -> Created: Dataset_Conclusion.csv (19 entries)


In [33]:
%pip install beautifulsoup4 requests pandas

  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [beautifulsoup4]m [beautifulsoup4]
Note: you may need to restart the kernel to use updated packages.


In [34]:
from bs4 import BeautifulSoup

In [35]:
PDF_DIR = "Annane_Papers_Collection"
OUTPUT_DIR = "Sepsis_Dataset" 
GROBID_URL = "http://localhost:8070/api/processFulltextDocument"

In [36]:
def process_pdf_with_grobid(pdf_path):
    """
    将 PDF 发送给 GROBID 服务，并获取 XML 结构化数据
    """
    try:
        files = {'input': open(pdf_path, 'rb')}
        # 请求 GROBID 解析全文
        response = requests.post(GROBID_URL, files=files, timeout=60)
        
        if response.status_code != 200:
            print(f"    ⚠️ GROBID 服务报错: {response.status_code}")
            return None
            
        return response.text
    except Exception as e:
        print(f"    ❌ 连接 GROBID 失败 (请确认 Docker 是否运行): {e}")
        return None

In [37]:
def parse_xml_structure(xml_content):
    """
    解析 TEI-XML，提取 Abstract, Method, Conclusion
    这是 GROBID 强大的地方，它已经帮我们打好标签了
    """
    soup = BeautifulSoup(xml_content, 'xml')
    
    data = {
        "Abstract": "",
        "Method": "",
        "Conclusion": ""
    }
    
    # 1. 提取 Abstract (XML 里直接有 <abstract> 标签)
    abstract_tag = soup.find("abstract")
    if abstract_tag:
        data["Abstract"] = abstract_tag.get_text(separator=" ", strip=True)
    
    # 2. 提取正文各章节 (XML 里是 <body> -> <div>)
    # GROBID 并不总是能完美标记 type="method"，所以我们结合 header 文本来做双重保险
    body = soup.find("body")
    if body:
        divs = body.find_all("div")
        for div in divs:
            # 获取该章节的标题
            head = div.find("head")
            header_text = head.get_text(strip=True).lower() if head else ""
            
            # 获取该章节的内容
            # 排除 head 标签，只取段落 p
            paragraphs = div.find_all("p")
            content = " ".join([p.get_text(separator=" ", strip=True) for p in paragraphs])
            
            # 智能分类逻辑
            if any(x in header_text for x in ['method', 'material', 'study design']):
                data["Method"] += "\n" + content
            
            elif any(x in header_text for x in ['conclusion', 'discussion', 'summary']):
                data["Conclusion"] += "\n" + content
            
            elif any(x in header_text for x in ['introduction', 'background', 'objective']):
                # 如果 Abstract 太短，可以把 Intro 补进去
                data["Abstract"] += "\n [Intro] " + content

    return data

In [38]:
def main():
    if not os.path.exists(OUTPUT_DIR):
        os.mkdir(OUTPUT_DIR)

    files = [f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")]
    results = []

    print(f"🚀 开始使用 GROBID 处理 {len(files)} 篇论文...")
    print("👉 请确保你已经在终端运行了: docker run -t --rm -p 8070:8070 lfoppiano/grobid:0.8.0")

    for i, filename in enumerate(files):
        print(f"[{i+1}/{len(files)}] 正在 AI 解析: {filename}")
        path = os.path.join(PDF_DIR, filename)
        
        # 1. 获取 XML
        xml_content = process_pdf_with_grobid(path)
        
        if xml_content:
            # 2. 解析 XML
            parsed_data = parse_xml_structure(xml_content)
            parsed_data['filename'] = filename
            results.append(parsed_data)
        else:
            print("    ❌ 跳过")

    # 保存
    df = pd.DataFrame(results)
    
    # 分三个文件保存，方便后续 NLP 训练
    if not df.empty:
        df[['filename', 'Abstract']].to_csv(os.path.join(OUTPUT_DIR, "Train_Abstract.csv"), index=False)
        df[['filename', 'Method']].to_csv(os.path.join(OUTPUT_DIR, "Train_Method.csv"), index=False)
        df[['filename', 'Conclusion']].to_csv(os.path.join(OUTPUT_DIR, "Train_Conclusion.csv"), index=False)
        print(f"\n✅ 处理完成！高质量数据已保存至 {OUTPUT_DIR}/")
    else:
        print("\n❌ 没有成功解析任何文件，请检查 GROBID 是否在运行。")

if __name__ == "__main__":
    main()

🚀 开始使用 GROBID 处理 19 篇论文...
👉 请确保你已经在终端运行了: docker run -t --rm -p 8070:8070 lfoppiano/grobid:0.8.0
[1/19] 正在 AI 解析: 34351722.pdf
[2/19] 正在 AI 解析: 35732834.pdf
[3/19] 正在 AI 解析: 26903338.pdf
[4/19] 正在 AI 解析: 41180093.pdf
[5/19] 正在 AI 解析: 26633262.pdf
[6/19] 正在 AI 解析: 29025194.pdf
[7/19] 正在 AI 解析: 33159530.pdf
[8/19] 正在 AI 解析: 37176780.pdf
[9/19] 正在 AI 解析: 33150472.pdf
[10/19] 正在 AI 解析: 34789298.pdf
[11/19] 正在 AI 解析: 30111341.pdf
[12/19] 正在 AI 解析: 10798738.pdf
[13/19] 正在 AI 解析: 38250247.pdf
[14/19] 正在 AI 解析: 33876268.pdf
[15/19] 正在 AI 解析: 30105022.pdf
[16/19] 正在 AI 解析: 39609770.pdf
[17/19] 正在 AI 解析: 37888913.pdf
[18/19] 正在 AI 解析: 23361625.pdf
[19/19] 正在 AI 解析: 39787606.pdf

✅ 处理完成！高质量数据已保存至 Sepsis_Dataset/


In [2]:
import pandas as pd
import json

In [4]:
df_abs = pd.read_csv("Sepsis_Dataset/Train_Abstract.csv")
df_met = pd.read_csv("Sepsis_Dataset/Train_Method.csv")
df_con = pd.read_csv("Sepsis_Dataset/Train_Conclusion.csv")

In [5]:
merged_data = []

In [ ]:
for index, row in df_con.iterrows():
    filename = row['filename']
    conclusion_text = row['Conclusion']
    
    # -------------------------------------------------------
    # 【核心清洗逻辑】
    # -------------------------------------------------------
    
    # 1. 检查 Claim 是否为空 (NaN, None, 或纯空字符串)
    if pd.isna(conclusion_text): 
        continue
    
    # 转成字符串并去除首尾空格
    clean_claim = str(conclusion_text).strip()
    
    # 再次检查：如果是 "nan" 字符串或者空串，跳过
    if clean_claim.lower() == 'nan' or clean_claim == "":
        continue

    # 2. 获取 Method 和 Abstract
    method_row = df_met[df_met['filename'] == filename]
    abstract_row = df_abs[df_abs['filename'] == filename]
    
    # 获取原始值
    method_val = method_row['Method'].values[0] if not method_row.empty else ""
    abstract_val = abstract_row['Abstract'].values[0] if not abstract_row.empty else ""

    # 3. 清洗 Context (强制转字符串，处理 NaN)
    clean_method = "" if pd.isna(method_val) else str(method_val).strip()
    clean_abstract = "" if pd.isna(abstract_val) else str(abstract_val).strip()
    
    # 如果 Method 里写的是 "nan"，也把它变空
    if clean_method.lower() == 'nan': clean_method = ""
    if clean_abstract.lower() == 'nan': clean_abstract = ""

    # 4. 最终检查：如果连背景材料都没有，没法分析，跳过
    if not clean_method and not clean_abstract:
        print(f"⚠️ 跳过无效数据 (无内容): {filename}")
        continue

    # 5. 构造最终条目 (确保所有字段都是 clean_xxx 字符串)
    entry = {
        "id": filename,
        "claim": clean_claim,
        # 拼接 Method 和 Background，中间加换行
        "evidence_context": f"Methodology:\n{clean_method}\n\nBackground:\n{clean_abstract}",
        "label": "Unknown"
    }
    merged_data.append(entry)

# 保存为 json (ensure_ascii=False 保证中文或其他字符正常显示)
output_file = "sepsis_dataset.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(merged_data, f, indent=4, ensure_ascii=False)

print("-" * 30)
print(f"✅ 最终清洗完毕！")
print(f"有效数据条数: {len(merged_data)}")
print(f"文件已覆盖保存为: {output_file}")

⚠️ 跳过无效数据 (无内容): 23361625.pdf
------------------------------
✅ 最终清洗完毕！
有效数据条数: 79
文件已覆盖保存为: sepsis_dataset.json
